<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/02_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 242, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 242 (delta 29), reused 11 (delta 4), pack-reused 190 (from 1)
Receiving objects: 100% (242/242), 193.13 KiB | 2.84 MiB/s, done.
Resolving deltas: 100% (133/133), done.


In [2]:
%cd /content/ML-Tech
!git pull origin main

/content/ML-Tech
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
import json

# each line of the bundled file is one {"user", "assistant"} chat example
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"loaded {len(chunks)} chunks")
print(chunks[0])

loaded 19 chunks
{'document': 'Personal attendance required.txt', 'title': 'Personal attendance required', 'url': 'https://www.general-security.gov.lb/en/posts/73', 'category': 'Personal attendance required', 'keywords': 'Lebanese citizens, minors, exemption from attendance, exemption from fees', 'section': 'Personal attendance required', 'text': 'Lebanese citizens that request a new passport should show up personally at the competent regional center of general security, according to their place of residence, having in hand an application that’s been filled, and certified by the competent mayor.\nMinors aged 7 years or younger have to accompany their parents to the mayor’s office, but don’t have to show up at the general security center. Both parents should sign a letter of consent at the mayor’s office, and convey their request to the general security. One of the parents can go on his own to the general security office, if the other parent signed the letter at the mayor’s office.\nMin

In [4]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.5 MB/s eta 0:00:00


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [6]:
texts = [chunk["text"] for chunk in chunks]
passages = ["passage: " + text for text in texts]

In [7]:
embeddings = model.encode(
    passages,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(19, 768)


In [8]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.write_index(index, "data/processed/passport_index.faiss")

In [9]:
faiss.write_index(
    index,
    "data/processed/passport_index.faiss"
)

In [10]:
import os

file_path = "data/processed/passport_index.faiss"
if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"Error: The file '{file_path}' does not exist. Please re-run the cell that creates it (cell `CMhSIbmozWxF`).")

The file 'data/processed/passport_index.faiss' exists.


In [11]:
import faiss

# Load the index to confirm it was created successfully
loaded_index = faiss.read_index("data/processed/passport_index.faiss")
print(f"Loaded FAISS index with {loaded_index.ntotal} vectors and dimension {loaded_index.d}.")

Loaded FAISS index with 19 vectors and dimension 768.


In [12]:
def retrieve(question, k=3):
    query_embedding = model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": chunks[idx]["document"],
            "section": chunks[idx]["section"],
            "text": chunks[idx]["text"]
        })

    return results

In [13]:
results = retrieve("Is it necessary for me to physically attend to get my passport?")

for result in results:
    print(result["score"])
    print(result["document"])
    print(result["section"])
    print(result["text"])
    print()

0.8466050624847412
Ex-porting Biometric Passport.txt
For the individual planning on shipping his passport with another traveler
1-   The owner needs to show up in person at the department of press – general security, with the traveler concerned, to convey the pre-mentioned request.
2-   The traveler has to have his airplane ticket in hand to underline the date of his departure, as well as proof of an entry visa and a stable residence in the country of destination
3-   The traveler is held responsible in case of losing the passport, on in case of any illegal use of the latter

0.8443633913993835
Personal attendance required.txt
Personal attendance required
Lebanese citizens that request a new passport should show up personally at the competent regional center of general security, according to their place of residence, having in hand an application that’s been filled, and certified by the competent mayor.
Minors aged 7 years or younger have to accompany their parents to the mayor’s offic

In [14]:
%cd /content/ML-Tech
!git pull origin main

/content/ML-Tech
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Already up to date.


In [15]:
!ls data/processed

chunks.json  passport_index.faiss
